[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Boutique_Fisica_Avanzada/N_Cuerpos_Gravedad.ipynb)



# Astrofísica: Simulación de N-Cuerpos Gravitacionales
Resolveremos dinámicamente la interacción gravitacional Newtoniana $1/r^2$ para un enjambre estelar cerrado utilizando integración simpléctica.

La gravedad es una fuerza de largo alcance y no apantallada. Todo interactúa con todo, escalando la complejidad computacional como $O(N^2)$.
Observaremos termodinámica estelar en vivo: un sistema frío colapsando violentamente, calentándose gravitacionalmente e interactuando caóticamente hasta formar un núcleo estelar denso rodeado por un halo termalizado (virialización).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Parámetros del cúmulo
N = 250        # Número de estrellas
G = 1.0        # Constante Gravitacional
softening = 0.1 # Suavizador para evitar singularidades 1/0 en choques directos
dt = 0.01      # Paso de integración

# Condiciones Iniciales (Nube esférica estática)
np.random.seed(42)
# Posiciones aleatorias en un círculo gaussiano
pos = np.random.randn(N, 2) * 2.0
# Velocidades inicialmente nulas (Colapso frío)
vel = np.zeros((N, 2))
# Masas aleatorias
mass = np.random.rand(N) * 2.0 + 0.1

def get_accelerations(p, m):
    """Calcula la aceleración gravitacional experimentada por cada partícula"""
    acc = np.zeros((N, 2))
    # O(N^2) bucle directo (Lento para N grandes, pero exacto)
    for i in range(N):
        for j in range(N):
            if i != j:
                dx = p[j, 0] - p[i, 0]
                dy = p[j, 1] - p[i, 1]
                inv_r3 = (dx**2 + dy**2 + softening**2)**(-1.5)
                acc[i, 0] += G * m[j] * dx * inv_r3
                acc[i, 1] += G * m[j] * dy * inv_r3
    return acc

# Aceleraciones iniciales
acc = get_accelerations(pos, mass)

# Visualización
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(8, 8))
scatter = ax.scatter(pos[:, 0], pos[:, 1], s=mass*10, c=mass, cmap='autumn', alpha=0.8, edgecolors='white', linewidth=0.5)
ax.set_xlim(-8, 8)
ax.set_ylim(-8, 8)
ax.set_title("Colapso de Cúmulo Estelar (N-Body Gravity)")
ax.axis('off')

def update(frame):
    global pos, vel, acc
    
    # Integrador Simpléctico Semi-Implícito de Euler (Kick-Drift)
    # o Leapfrog Verlet. Usamos Semi-Implícito aquí:
    vel += acc * dt
    pos += vel * dt
    acc = get_accelerations(pos, mass)
    
    scatter.set_offsets(pos)
    return scatter,

# Para animar en Jupyter:
# anim = FuncAnimation(fig, update, frames=300, interval=20, blit=True)
# HTML(anim.to_jshtml())

# Simulamos 50 pasos sin animar para mostrar la foto en progreso
for _ in range(50):
    vel += acc * dt
    pos += vel * dt
    acc = get_accelerations(pos, mass)
scatter.set_offsets(pos)
plt.show()
print("Las estrellas comienzan a caer dramáticamente hacia el centro de masas, ganando energía cinética (calor) a expensas de la energía potencial gravitatoria.")